# Population ESP composites: selection audit
Run on Katana first. This produces a frozen manifest for **AE/CE × planetary/topographic** composites. The initial product uses valid ESP geometry and complete profiles, not a claim of validated reconstruction skill. Existing final surface tables contain no interpolation provenance; this limitation is recorded for every row. No outcome-based tilt filter is used.

Quality thresholds below are provisional and transparent. Inspect retention before interpreting results. Mixed days remain in the audit but are outside the initial four composites.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
HERE = Path.cwd().resolve()
ANALYSIS = next((p for p in (HERE, *HERE.parents) if (p/'seacofs_tilt_tools.py').exists()), None)
if ANALYSIS is None:
    raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subdirectory')
WORK = ANALYSIS/'esp_population_composites'
for p in (ANALYSIS, WORK):
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import seacofs_tilt_tools as tilt
import population_tools as pop
pd.set_option('display.max_columns', 60)


In [ ]:
# Exact cached levels nearest these targets; no vertical interpolation.
TARGET_DEPTHS_M = (0., 200., 500.)
MAX_DEPTH_DISTANCE_M = 60.
DOMINANCE_FACTOR = 2.
MIN_SLOPE = 1e-4  # m/m, net core-mean bathymetric gradient
MIN_SLOPE_COHERENCE = 0.5
MIN_SLOPE_VALID_FRACTION = 0.8
HORIZONTAL_UNITS = 'Rc'  # one reference Rc for the whole column
HALF_WIDTH = 3.0        # use e.g. 100 if units='km'
GRID_STEP = 0.2         # use e.g. 5 if units='km'
MASK_OCEAN = True      # native wet/bathymetric support at each transformed point
N_BOOT = 500
MIN_EDDIES = 20        # pointwise display threshold, not a guarantee of precision
SEED = 731
ESP_ROOT = '/home/z5297792/ESP_zonodo'
OUTPUT_ROOT = Path('/srv/scratch/z5297792/SEACOFS_26yr_eddy_dataset_modular/esp_population_composites')
paths = tilt.Paths()
PV_PATH = tilt.DEFAULT_DEPTH_PV_ROOT/tilt.DEPTH_PV_SNAPSHOT_NAME


In [ ]:
surface = tilt.read_table(paths.eddies)
vertical = tilt.load_vert(paths)
pv = pd.read_parquet(PV_PATH)
pop.unique(surface, pop.KEYS)
pop.unique(vertical, pop.KEYS+['Depth'])
pop.unique(pv, pop.KEYS)
depths = pop.select_depths(vertical, TARGET_DEPTHS_M, MAX_DEPTH_DISTANCE_M)
print('Requested:', TARGET_DEPTHS_M, 'Exact fitted levels:', depths)
# This reference depth is used for EVERY eddy-day, even if some profiles have shallower levels.
print('Common reference depth:', depths[0])
catalogue = vertical.groupby('Depth').size().rename('eddy_days').to_frame()
display(catalogue)
audit, fitted = pop.audit_population(surface, vertical, pv, depths, DOMINANCE_FACTOR)
grid = tilt.load_grid(paths.grid, paths.z_r)
audit = pop.add_frames(audit, grid, min_slope=MIN_SLOPE,
                       min_coherence=MIN_SLOPE_COHERENCE, min_fraction=MIN_SLOPE_VALID_FRACTION)


## Attrition and population balance
Inspect both numbers of eddies and days. Basic validity exclusions can overlap. Missing deep profiles may preferentially remove shallower eddies; compare with a shallower run. Slope-direction failures only exclude slope-aligned composites. Neither tilt magnitude nor agreement with a hypothesised direction determines membership.

In [ ]:
display(audit.groupby(['Cyc','regime'], observed=True).agg(
    candidate_days=('Day','size'), candidate_eddies=('Eddy','nunique'),
    basic_eligible_days=('eligible','sum'), selected_days=('selected','sum')))
display(audit.exclusion_reason.value_counts(dropna=False).rename('days'))
selected = audit.loc[audit.selected].copy()
if selected.empty:
    raise ValueError('No selected eddy-days; inspect the audit and depth catalogue')
display(selected.groupby('group').agg(eddies=('Eddy','nunique'),days=('Day','size')))
display(selected.groupby(['group','Eddy']).size().groupby('group').describe())
columns = [c for c in ['lat','lon','Rc','Ro','h','slope_strength','slope_coherence'] if c in selected]
display(selected.groupby('group')[columns].agg(['median','min','max']))
fig, ax = plt.subplots(figsize=(8,6), constrained_layout=True)
for group, part in selected.groupby('group'):
    ax.scatter(part.lon, part.lat, s=2, alpha=.15, label=group)
ax.set(xlabel='Longitude', ylabel='Latitude', title='Selected eddy-day geography')
ax.legend(markerscale=4)
plt.show()


## Freeze selection and source metadata
Outputs use a configuration/source-metadata hash. A separate hash is generated for different depths, thresholds, normalisation or inputs. Full source-file contents are not hashed. Reconstruction will verify the stored file sizes and modification times. The audit records the processed-surface provenance limitation; it does not invent an observed-only flag.

In [ ]:
config = dict(depths=depths.tolist(), target_depths=list(TARGET_DEPTHS_M),
              dominance=DOMINANCE_FACTOR, min_slope=MIN_SLOPE,
              min_slope_coherence=MIN_SLOPE_COHERENCE,
              min_slope_valid_fraction=MIN_SLOPE_VALID_FRACTION,
              units=HORIZONTAL_UNITS, half_width=HALF_WIDTH, grid_step=GRID_STEP,
              mask_ocean=MASK_OCEAN, n_boot=N_BOOT, min_eddies=MIN_EDDIES,
              seed=SEED, esp_root=ESP_ROOT,
              velocity_convention='existing SEACOFS ESP: east/north output; model-grid x/y inputs')
run_id, provenance = pop.fingerprint(config, [paths.eddies, paths.vert, PV_PATH, paths.grid, paths.z_r, Path(ESP_ROOT)/'functions.py'])
run = OUTPUT_ROOT/run_id
run.mkdir(parents=True, exist_ok=True)
audit.to_parquet(run/'audit.parquet', index=False)
selected.to_parquet(run/'selected.parquet', index=False)
fitted.merge(selected[pop.KEYS], on=pop.KEYS, validate='many_to_one').to_parquet(run/'profiles.parquet', index=False)
(run/'provenance.json').write_text(json.dumps(provenance, indent=2))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUTPUT_ROOT/'latest_run.txt').write_text(str(run))
print('Frozen run:', run)
print('Next: 01_polarity_regime_composites.ipynb')
